### Connect to PostgreSQL Database

To connect to a PostgreSQL database from Python, we typically use the `psycopg2` library. First, we need to install it.

In [1]:
# Install the psycopg2 library
%pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 36.8 MB/s eta 0:00:00


Now, we can use `psycopg2` to connect to your PostgreSQL server. I'll use the connection string you provided. It's a good practice to handle potential connection errors using a `try-except` block.

In [2]:
import psycopg2

# Your provided PostgreSQL connection string
# NOTE: For production environments, sensitive credentials like this should be stored securely
# (e.g., environment variables, Google Cloud Secret Manager, or Colab's secrets manager).
# For this example, we'll use it directly as provided.
connection_string = ""

conn = None
cursor = None

try:
    # Establish the connection
    conn = psycopg2.connect(connection_string)
    print("Successfully connected to PostgreSQL database!")

    # Create a cursor object
    cursor = conn.cursor()

    # Execute a simple query to verify the connection
    cursor.execute("SELECT version();")
    db_version = cursor.fetchone()
    print(f"PostgreSQL database version: {db_version[0]}")

except psycopg2.Error as e:
    print(f"Error connecting to PostgreSQL database: {e}")

finally:
    # Close the cursor and connection
    if cursor:
        cursor.close()
    if conn:
        conn.close()
    print("Connection closed.")

Successfully connected to PostgreSQL database!
PostgreSQL database version: PostgreSQL 17.10 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.1 20260123 (Red Hat 15.2.1-7), 64-bit
Connection closed.


### Import SQL Dump File

To import a `.sql` dump file into PostgreSQL, we typically use the `psql` command-line tool. However, since we are in a Python environment, we can download the file and then use `psycopg2` to read and execute the SQL commands within the Python script.

First, let's download the `dvdrental.sql` file from the provided URL.

In [3]:
import requests

sql_dump_url = "https://raw.githubusercontent.com/neondatabase/postgres-sample-dbs/refs/heads/main/dvdrental.sql"
output_filename = "dvdrental.sql"

try:
    response = requests.get(sql_dump_url)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    with open(output_filename, 'wb') as f:
        f.write(response.content)
    print(f"Successfully downloaded '{output_filename}'")

except requests.exceptions.RequestException as e:
    print(f"Error downloading SQL dump file: {e}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Successfully downloaded 'dvdrental.sql'


Now that the SQL file is downloaded, we can read its content and execute the commands using `psycopg2`. We will establish a new connection and execute the SQL script. This might take a few moments as the `dvdrental.sql` file contains a schema and data for a sample database.

In [8]:
import psycopg2
import subprocess
import os

# Your provided PostgreSQL connection string
# NOTE: For production environments, sensitive credentials like this should be stored securely.
# For this example, we'll use it directly as provided by user.

output_filename = "dvdrental.sql" # This comes from the previous cell

conn = None
cursor = None

try:
    print(f"Attempting to import '{output_filename}' using psql...")

    # Construct the psql command
    # -X disables client-side commands to avoid conflicts
    # -v ON_ERROR_STOP=1 makes psql exit immediately on error
    # -d specifies the database connection string
    # -f specifies the input file
    psql_command = [
        "psql",
        "-v", "ON_ERROR_STOP=1",
        "-d", connection_string,
        "-f", output_filename
    ]

    # Execute the psql command
    # We're capturing stdout and stderr to display potential issues.
    # The database password is part of the connection_string, so it's passed via -d.
    result = subprocess.run(psql_command, capture_output=True, text=True, check=False)

    if result.returncode == 0:
        print("SQL dump imported successfully using psql!")
        print("psql stdout:")
        print(result.stdout)
    else:
        print(f"Error importing SQL dump using psql. Exit code: {result.returncode}")
        print("psql stdout:")
        print(result.stdout)
        print("psql stderr:")
        print(result.stderr)
        # Raise an exception to indicate failure and trigger the outer try-except's rollback
        raise Exception("psql import failed")

    # --- Verify the import by connecting with psycopg2 and querying ---
    print("\nVerifying import by connecting and querying the 'actor' table...")
    conn = psycopg2.connect(connection_string)
    cursor = conn.cursor()

    cursor.execute("SELECT COUNT(*) FROM actor;")
    actor_count = cursor.fetchone()[0]
    print(f"Number of actors in the 'actor' table: {actor_count}")

    conn.commit() # Commit any implicit transactions if necessary (though psql handles its own)
    print("Verification successful.")

except psycopg2.Error as e:
    print(f"Database connection or query error: {e}")
    if conn:
        conn.rollback()
        print("Transaction rolled back.")
except Exception as e:
    print(f"An unexpected error occurred during psql execution or verification: {e}")
    if conn:
        conn.rollback()
        print("Transaction rolled back due to unexpected error.")
finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()
    print("Connection closed.")

Attempting to import 'dvdrental.sql' using psql...
SQL dump imported successfully using psql!
psql stdout:
SET
SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
CREATE TYPE
CREATE DOMAIN
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE FUNCTION
CREATE SEQUENCE
SET
CREATE TABLE
CREATE FUNCTION
CREATE AGGREGATE
CREATE SEQUENCE
CREATE TABLE
CREATE SEQUENCE
CREATE TABLE
CREATE SEQUENCE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE VIEW
CREATE SEQUENCE
CREATE TABLE
CREATE SEQUENCE
CREATE TABLE
CREATE SEQUENCE
CREATE TABLE
CREATE VIEW
CREATE VIEW
CREATE SEQUENCE
CREATE TABLE
CREATE SEQUENCE
CREATE TABLE
CREATE VIEW
CREATE SEQUENCE
CREATE TABLE
CREATE SEQUENCE
CREATE TABLE
CREATE VIEW
CREATE SEQUENCE
CREATE TABLE
CREATE SEQUENCE
CREATE TABLE
CREATE VIEW
CREATE VIEW
COPY 200
COPY 603
COPY 16
COPY 600
COPY 109
COPY 599
COPY 1000
COPY 5462
COPY 1000
COPY 4581
COPY 6
COPY 14596
COPY 16044
COPY 2
COPY 2
 s

### Verify Database Schema - List Table Names

Let's connect to the database again and query the `information_schema.tables` to list all tables that were created during the import of `dvdrental.sql`. This will confirm that the schema was correctly applied.

In [9]:
import psycopg2

conn = None
cursor = None

try:
    conn = psycopg2.connect(connection_string)
    cursor = conn.cursor()

    # Query to get all table names in the 'public' schema (default for dvdrental)
    cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name;")

    print("Tables in 'public' schema:")
    table_names = cursor.fetchall()
    if table_names:
        for table in table_names:
            print(f"- {table[0]}")
    else:
        print("No tables found in the 'public' schema.")

except psycopg2.Error as e:
    print(f"Error connecting to or querying the database: {e}")
finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()
    print("Connection closed.")

Tables in 'public' schema:
- actor
- actor_info
- address
- category
- city
- country
- customer
- customer_list
- film
- film_actor
- film_category
- film_list
- inventory
- language
- nicer_but_slower_film_list
- payment
- rental
- sales_by_film_category
- sales_by_store
- staff
- staff_list
- store
Connection closed.


### Querying the `film` table

Let's run a sample query to retrieve the first 5 rows from the `film` table to ensure that the data was also imported correctly.

In [10]:
import psycopg2

conn = None
cursor = None

try:
    conn = psycopg2.connect(connection_string)
    cursor = conn.cursor()

    # Select the first 5 rows from the 'film' table
    query = "SELECT * FROM film LIMIT 5;"
    cursor.execute(query)

    print("First 5 rows from the 'film' table:")
    # Get column names
    column_names = [desc[0] for desc in cursor.description]
    print(column_names)

    # Fetch and print rows
    for row in cursor.fetchall():
        print(row)

except psycopg2.Error as e:
    print(f"Error connecting to or querying the database: {e}")
finally:
    if cursor:
        cursor.close()
    if conn:
        conn.close()
    print("Connection closed.")

First 5 rows from the 'film' table:
['film_id', 'title', 'description', 'release_year', 'language_id', 'rental_duration', 'rental_rate', 'length', 'replacement_cost', 'rating', 'last_update', 'special_features', 'fulltext']
(133, 'Chamber Italian', 'A Fateful Reflection of a Moose And a Husband who must Overcome a Monkey in Nigeria', 2006, 1, 7, Decimal('4.99'), 117, Decimal('14.99'), 'NC-17', datetime.datetime(2013, 5, 26, 14, 50, 58, 951000), ['Trailers'], "'chamber':1 'fate':4 'husband':11 'italian':2 'monkey':16 'moos':8 'must':13 'nigeria':18 'overcom':14 'reflect':5")
(384, 'Grosse Wonderful', 'A Epic Drama of a Cat And a Explorer who must Redeem a Moose in Australia', 2006, 1, 5, Decimal('4.99'), 49, Decimal('19.99'), 'R', datetime.datetime(2013, 5, 26, 14, 50, 58, 951000), ['Behind the Scenes'], "'australia':18 'cat':8 'drama':5 'epic':4 'explor':11 'gross':1 'moos':16 'must':13 'redeem':14 'wonder':2")
(8, 'Airport Pollock', 'A Epic Tale of a Moose And a Girl who must Confront

### Exporting 'film' table to Pandas DataFrame

Now, let's load the entire `film` table into a pandas DataFrame for further analysis. We'll use `pandas.read_sql_query` which simplifies the process of fetching data directly into a DataFrame.

In [11]:
import pandas as pd
import psycopg2

conn = None

try:
    conn = psycopg2.connect(connection_string)

    # Use pandas.read_sql_query to directly load data into a DataFrame
    # We'll select all rows from the 'film' table.
    df_film = pd.read_sql_query("SELECT * FROM film;", conn)

    print("DataFrame created successfully. Displaying the first 5 rows:")
    display(df_film.head())

except psycopg2.Error as e:
    print(f"Error connecting to or querying the database: {e}")
finally:
    if conn:
        conn.close()
    print("Connection closed.")

/tmp/ipykernel_7883/3419852801.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_film = pd.read_sql_query("SELECT * FROM film;", conn)


DataFrame created successfully. Displaying the first 5 rows:


,film_id,title,description,release_year,language_id,rental_duration,rental_rate,length,replacement_cost,rating,last_update,special_features,fulltext
0,133,Chamber Italian,A Fateful Reflection of a Moose And a Husband ...,2006,1,7,4.99,117,14.99,NC-17,2013-05-26 14:50:58.951,[Trailers],'chamber':1 'fate':4 'husband':11 'italian':2 ...
1,384,Grosse Wonderful,A Epic Drama of a Cat And a Explorer who must ...,2006,1,5,4.99,49,19.99,R,2013-05-26 14:50:58.951,[Behind the Scenes],'australia':18 'cat':8 'drama':5 'epic':4 'exp...
2,8,Airport Pollock,A Epic Tale of a Moose And a Girl who must Con...,2006,1,6,4.99,54,15.99,R,2013-05-26 14:50:58.951,[Trailers],'airport':1 'ancient':18 'confront':14 'epic':...
3,98,Bright Encounters,A Fateful Yarn of a Lumberjack And a Feminist ...,2006,1,4,4.99,73,12.99,PG-13,2013-05-26 14:50:58.951,[Trailers],'boat':20 'bright':1 'conquer':14 'encount':2 ...
4,1,Academy Dinosaur,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,6,0.99,86,20.99,PG,2013-05-26 14:50:58.951,"[Deleted Scenes, Behind the Scenes]",'academi':1 'battl':15 'canadian':20 'dinosaur...


Connection closed.


### Install PostgreSQL Client Tools

The error `[Errno 2] No such file or directory: 'psql'` indicates that the `psql` command-line tool is not available in the environment. We need to install the PostgreSQL client packages.

In [7]:
# Install PostgreSQL client packages
# This provides the 'psql' command-line tool
!apt-get update
!apt-get install -y postgresql-client

print("PostgreSQL client tools installed. Please re-run the cell above to import the SQL dump.")

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.4 MB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,303 kB]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:14 